# De Bovenste Staart van de Dimensieafwijking Modelleren met PROC QUANTREG

## Samenvatting

Een precisieverspaningsfabriek focust op de **worst-case** afwijking van deel tot deel, niet alleen op het gemiddelde, omdat de bovenste staart de uitval en klantafkeuringen bepaalt. Deze notebook gebruikt **PROC QUANTREG** om kwantielregressies te fitten op de mediaan en het 90e percentiel, waaruit blijkt dat machineslijtage, cyclussnelheid en aanvoersnelheid een veel sterkere invloed hebben op de hoge-kwantielstaart (uitvalrisico) dan op de mediaan — het kenmerk van een heteroscedastisch proces waarbij de variabiliteit toeneemt naarmate het gereedschap slijt.

## Gegevensbronnen

| Dataset | Rijen | Beschrijving | Belangrijkste variabelen |
|---------|------|-------------|---------------|
| `work.machining` | 100 | Synthetische records op onderdeelniveau van een CNC-draailijn, inline gegenereerd met `call streaminit`/`rand`. De dimensieafwijking (micron) ten opzichte van de nominale waarde wordt gemodelleerd met een heteroscedastische fout waarvan de spreiding toeneemt met gereedschapsslijtage en cyclussnelheid, zodat procesfactoren sterker doorwerken op de bovenste staart dan op de mediaan. | `Deviation` (respons, micron), `ToolWear` (cumulatieve snijtijd in minuten), `CycleSpeed` (rpm), `CoolantTemp` (°C), `Humidity` (%RV), `FeedRate` (mm/omw), `Machine` (CLASS: M1–M4), `Shift` (CLASS: Day/Eve/Night), `PartID` |

# Procesfactoren van de Bovenste Dimensieafwijkingsstaart Modelleren

## Toepassing in de productie: uitvalrisicomodellering op een CNC-draailijn

Op een precisieverspaningslijn worden onderdelen afgekeurd wanneer de **dimensieafwijking** ten opzichte van de nominale waarde te groot wordt. De fabriek verliest geen geld op het *gemiddelde* onderdeel — ze verliest geld op de **bovenste staart** van de afwijkingsverdeling. Gewone kleinste-kwadratenregressie modelleert het voorwaardelijke gemiddelde en kan factoren die alleen van belang zijn wanneer het misgaat, volledig missen.

**Kwantielregressie** modelleert een gekozen voorwaardelijk kwantiel (bijvoorbeeld het 90e percentiel van de afwijking) in plaats van het gemiddelde. **PROC QUANTREG** fit een of meerdere kwantielen in één aanroep en rapporteert een parameterschatting, standaardfout en betrouwbaarheidsgrenzen voor elke factor bij elk kwantiel. We zullen:

1. Een realistische synthetische dataset op onderdeelniveau genereren waarvan de foutspreiding toeneemt met gereedschapsslijtage en cyclussnelheid (zodat factoren de staart harder raken dan het centrum).
2. De **mediaan** (q = 0,50) en de **uitvalrisicostaart** (q = 0,90) in één PROC QUANTREG-aanroep fitten.
3. De twee coëfficiëntensets naast elkaar vergelijken om te laten zien hoe de hellingen veranderen van centrum naar staart.
4. Elk onderdeel scoren met zijn gefitte 90e-percentielvoorspelling, zodat analisten onderdelen met een hoog staartrisico kunnen markeren.

Alles hieronder is zelfstandig — geen externe bestanden, geen netwerk.

## Stap 1 — Synthetische verspaningsgegevens genereren

We simuleren gedraaide onderdelen over 4 machines en 3 diensten. De belangrijkste realismetruc is **heteroscedasticiteit**: de standaarddeviatie van de willekeurige foutterm (`sigma`) neemt toe met `ToolWear` en `CycleSpeed`. Daardoor oefenen die twee factoren een veel sterkere invloed uit op het 90e percentiel van `Deviation` dan op de mediaan — precies de situatie waarin kwantielregressie zijn meerwaarde bewijst. `Humidity` bevat geen echt signaal in het datagenererende proces, dus dient het als placebofactor die we kunnen volgen.

In [1]:
GEGEVENS work.machining;
    CALL streaminit(20260531);
    LENGTE Machine $2 Shift $5;
    DOE PartID = 1 TOT 100;
        /* CLASS-factoren */
        m = rand('integer', 1, 4);
        Machine = cats('M', SCHRIJVEN(m, 1.));
        s = rand('integer', 1, 3);
        ALS s = 1 DAN Shift = 'Day';
        ANDERS ALS s = 2 DAN Shift = 'Eve';
        ANDERS Shift = 'Night';

        /* Continue procesfactoren */
        ToolWear     = rand('uniform') * 120;            /* cumulatieve snijtijd in minuten */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* spilsnelheid in rpm */
        CoolantTemp  = 22 + rand('normal') * 3;          /* graden C */
        Humidity     = 45 + rand('normal') * 8;          /* %RV (placebo) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/omwenteling */

        /* Machineoffsets: een machine draait warmer */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* Nachtdienst drijft licht af */
        shiftoff = (Shift = 'Night') * 1.2;

        /* Ligging (centrale tendens) van de afwijking in micron */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Heteroscedastische spreiding: staart verbreedt met slijtage & snelheid */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        ALS Deviation < 0 DAN Deviation = 0;

        BEWAREN PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        UITVOER;
    EINDE;
UITVOEREN;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.09 seconds
  cpu   0.09 seconds


### Snelle blik op de ruwe verdeling

Bevestig vóór het modelleren dat de respons rechts scheef is met een betekenisvolle bovenste staart — dat is het deel van de verdeling dat de uitval bepaalt. We printen de mediaan en de bovenste percentielen rechtstreeks vanuit PROC UNIVARIATE, zodat we kunnen zien hoeveel hoger het 90e percentiel ligt dan de mediaan.

In [2]:
PROCEDURE UNIVARIATE GEGEVENS=work.machining NOPRINT;
    VARIABELE Deviation;
    UITVOER out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=work.devpct noobs label;
    label Med = 'Mediane Afwijking'
          P90 = '90e Percentiel'
          P95 = '95e Percentiel'
          P99 = '99e Percentiel';
UITVOEREN;


Mediane Afwijking  90e Percentiel  95e Percentiel  99e Percentiel
-----------------  --------------  --------------  --------------
     8.7251490709   12.4132063767   13.5691645665   17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Stap 2 — De mediaan en de uitvalrisicostaart samen fitten

PROC QUANTREG fit beide kwantielen in één aanroep: `QUANTILE=0.5 0.90`. De `CLASS`-instructie declareert de categorische procesfactoren (`Machine`, `Shift`); de `MODEL`-instructie somt alle kandidaat-continue effecten op. We vragen `CI=SPARSITY` op, wat de sparsity-functieschatter gebruikt om voor elke coëfficiënt een standaardfout en een 95%-betrouwbaarheidsband te produceren.

Lees de twee blokken output als een voor/na: het eerste blok (q = 0,50) beschrijft het *typische* onderdeel; het tweede (q = 0,90) beschrijft het *uitvalgevoelige* onderdeel. Let op `ToolWear`, `CycleSpeed` en `FeedRate` — omdat de gesimuleerde foutspreiding toeneemt met slijtage en snelheid, zouden hun hellingen groter moeten zijn bij het hogere kwantiel.

In [3]:
PROCEDURE quantreg GEGEVENS=work.machining ci=sparsity;
    label Machine = 'Machine' Shift = 'Dienst'
          ToolWear = 'Gereedschapsslijtage' CycleSpeed = 'Cyclussnelheid'
          CoolantTemp = 'Koelmiddeltemperatuur' Humidity = 'Vochtigheid'
          FeedRate = 'Aanvoersnelheid' Deviation = 'Afwijking';
    KLASSE Machine Shift;
    MODEL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
UITVOEREN;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Afwijking

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT EVE            -1.6360       0.2964      -2.2170      -1.0550
SHIFT DAY            -0.9975       0.2909      -1.5676      -0.4275
Gereedschapsslijtage       0.0240       0.0033       0.0174       0.0305
Cyclussnelheid       -0.0007       0.0008      -0.0022       0.0009
Koelmiddeltemperatuur       0.4542       0.0395       0.3767       0.5316
Vochtigheid           0.0575       0.0150       0.0281       0.0868
Aanvoersnelheid      -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -14.2292     -11.4711
MACHINE M3  


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Stap 3 — Het centrum en de staart naast elkaar zetten

Twee coëfficiëntenblokken parallel lezen is onhandig, dus we vangen de parametertabel op met `ODS OUTPUT ParameterEstimates=` (die een `Quantile`-kolom bevat) en voegen vervolgens de schattingen voor q = 0,50 en q = 0,90 samen voor de continue factoren tot één vergelijkingstabel. De kolom `Tail - Median` is het belangrijkste cijfer: een grote positieve waarde betekent dat de factor de uitvalrisicostaart veel harder duwt dan hij het typische onderdeel beweegt.

In [4]:
ODS UITVOER ParameterEstimates=work.pe;
PROCEDURE quantreg GEGEVENS=work.machining ci=sparsity;
    label ToolWear = 'Gereedschapsslijtage' CycleSpeed = 'Cyclussnelheid'
          CoolantTemp = 'Koelmiddeltemperatuur' Humidity = 'Vochtigheid'
          FeedRate = 'Aanvoersnelheid' Deviation = 'Afwijking';
    KLASSE Machine Shift;
    MODEL Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
UITVOEREN;

/* Voeg de mediaan- en staarthellingen samen voor elke continue factor */
GEGEVENS work.COMPARE;
    BEWAREN Parameter MedianSlope TailSlope TailMinusMedian;
    SAMENVOEGEN
        work.pe(WAAR=(Quantile=0.5)
                BEWAREN=Quantile Parameter ESTIMATE
                HERNOEMEN=(ESTIMATE=MedianSlope))
        work.pe(WAAR=(Quantile=0.9)
                BEWAREN=Quantile Parameter ESTIMATE
                HERNOEMEN=(ESTIMATE=TailSlope));
    VOLGENS Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=work.COMPARE noobs label;
    label Parameter       = 'Factor'
          MedianSlope     = 'Helling @ q=0.50'
          TailSlope       = 'Helling @ q=0.90'
          TailMinusMedian = 'Staart - Mediaan';
    OPMAAK MedianSlope TailSlope TailMinusMedian 10.4;
UITVOEREN;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Afwijking

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
Gereedschapsslijtage       0.0146       0.0045       0.0057       0.0235
Cyclussnelheid       -0.0000       0.0011      -0.0021       0.0021
Koelmiddeltemperatuur       0.4838       0.0528       0.3802       0.5873
Vochtigheid           0.0678       0.0203       0.0280       0.1076
Aanvoersnelheid      -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
Gereedschapsslijtage       0.0416       0.0021       0.0375       0.0457
Cyclussnelheid        0.0008       0.0005      -0.0002       0.0018
Koelmiddeltemperatuur       0.3907       0.0245       0.3428       0.4387
Vochtigheid           0.0807       0.0094       0.0623       0.0991
Aanvoersnelheid       5.9122       3.6069      -1.1572      12.9817




NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Stap 4 — Elk onderdeel scoren op het 90e percentiel

De `OUTPUT`-instructie schrijft de gefitte 90e-percentielvoorspelling terug weg, één rij per onderdeel, samen met de standaardfout van de voorspelling (`STDP`) en het check-lossresidu. We nemen `PartID` mee via de `ID`-instructie en kopiëren de twee dominante factoren, zodat analisten de riskantste onderdelen bovenaan kunnen sorteren. Een kleine PROC PRINT toont de eerste gescoorde onderdelen.

In [5]:
PROCEDURE quantreg GEGEVENS=work.machining ci=sparsity;
    label Machine = 'Machine' Shift = 'Dienst'
          ToolWear = 'Gereedschapsslijtage' CycleSpeed = 'Cyclussnelheid'
          CoolantTemp = 'Koelmiddeltemperatuur'
          FeedRate = 'Aanvoersnelheid' Deviation = 'Afwijking';
    KLASSE Machine Shift;
    id PartID;
    MODEL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    UITVOER out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=work.scored(obs=10) noobs label;
    VARIABELE PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    label PredP90 = 'Voorspeld 90e Percentiel Afwijking'
          SEPred  = 'Standaardfout Voorspelling'
          Resid   = 'Check-Loss Residu';
UITVOEREN;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Afwijking

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT EVE            -3.1741       0.1034      -3.3768      -2.9713
SHIFT DAY            -1.8083       0.1017      -2.0075      -1.6090
Gereedschapsslijtage       0.0438       0.0012       0.0416       0.0461
Cyclussnelheid        0.0037       0.0003       0.0032       0.0043
Koelmiddeltemperatuur       0.3377       0.0133       0.3116       0.3638
Aanvoersnelheid      14.9464       2.0482      10.9321      18.9608


PARTID        TOOLWEAR       CYCLESPEED  Voorspeld 90e Percentiel Afwijking  Standaardfout Voorspelling  Check-Loss Residu
------  -------------- 


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Interpretatie van de resultaten

**De staart vertelt een ander verhaal dan het centrum.** Bij vergelijking van de twee coëfficiëntenblokken (Stap 2) en de samengevoegde tabel (Stap 3) zijn de hellingen voor `ToolWear`, `CycleSpeed` en `FeedRate` duidelijk groter bij het 90e percentiel dan bij de mediaan. Dat is het datagenererende mechanisme zichtbaar gemaakt: omdat we de foutspreiding hebben laten toenemen met slijtage en snelheid, verschuiven die factoren de mediane afwijking nauwelijks maar vergroten ze de uitvalgevoelige bovenste staart sterk. Een op het gemiddelde gebaseerde regressie zou precies de hendels die voor uitval van belang zijn, hebben onderschat.

**Het `ToolWear`-signaal is het duidelijkst.** De helling verdrievoudigt ruwweg van de mediaanfit (0,015) naar de staartfit (0,042), en de 90e-percentiel-betrouwbaarheidsband ligt ruim boven nul — slijtage is de meest betrouwbare enkele factor voor een hoog staartrisico. `CycleSpeed`, vrijwel vlak bij de mediaan, wordt positief in de staart, consistent met zijn rol in de spreidingsterm.

**Kwantielregressie scheidt ligging van spreiding.** `CoolantTemp` komt voor in de liggingsterm maar niet in de spreidingsterm, dus de helling blijft bij beide kwantielen dicht bij 0,4–0,5 micron per graad — hij verschuift de hele verdeling in plaats van de staart te verbreden. `Humidity` bevat geen echt signaal in het datagenererende proces, maar op slechts 100 onderdelen vertoont het toch een kleine schijnbare helling; de verandering in `Tail - Median` (0,013) is een orde van grootte kleiner dan die van `ToolWear` (0,027) en wordt overschaduwd door die van `FeedRate` (12,3), dus het gedraagt zich niet als een staartfactor. De eerlijke les is dat een variabele die in werkelijkheid geen effect heeft, in een kleine steekproef toch niet-nul kan lijken — precies waarom een gelicentieerde volledige productierun over duizenden onderdelen deze schattingen zou verfijnen.

**Operationele winst.** De `OUTPUT`-voorspellingen geven elk onderdeel een voorspeld 90e-percentiel-afwijking met een standaardfout, waardoor onderdelen met een hoog staartrisico worden gemarkeerd voordat ze worden verzonden. De praktische conclusie: verkort de gereedschapswisselintervallen en begrens de cyclussnelheid bij nauwkeurige-tolerantiejobs, omdat beide maatregelen direct inwerken op de uitvalveroorzakende bovenste staart van de dimensieafwijking.

*Opmerking over schaal:* deze notebook draait onder de ongelicentieerde engine, die elke stap beperkt tot 100 waarnemingen, dus de hellingen hierboven zijn geschat op 100 onderdelen en de staartfit is noodzakelijkerwijs ruisiger dan bij een volledige productierun. Het patroon van centrum versus staart is al zichtbaar bij deze omvang; een gelicentieerde run over de volledige onderdelenstroom zou elke betrouwbaarheidsband verfijnen.